<a href="https://colab.research.google.com/github/rxnu/LLM-Project/blob/main/2_reprensentation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

# vectorizer with basic cleaning
tfidf = TfidfVectorizer(
    lowercase=True,
    stop_words='english',
    max_features=20_000
)

# fit on train reviews, transform both splits
X_train = tfidf.fit_transform(train_df['text'])
X_test  = tfidf.transform(test_df['text'])

print(f"Vocabulary size: {len(tfidf.vocabulary_)}")

Vocabulary size: 20000


In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report

# train
clf = LogisticRegression(max_iter=1000)
clf.fit(X_train, train_df['label'])

# predict
y_pred = clf.predict(X_test)
acc = accuracy_score(test_df['label'], y_pred)
print(f"Accuracy: {acc:.4%}\n")
print(classification_report(test_df["label"], y_pred, target_names=["neg","pos"]))

Accuracy: 87.9960%

              precision    recall  f1-score   support

         neg       0.88      0.88      0.88     12500
         pos       0.88      0.88      0.88     12500

    accuracy                           0.88     25000
   macro avg       0.88      0.88      0.88     25000
weighted avg       0.88      0.88      0.88     25000



In [ ]:

# Top Predictive Features (Logistic Regression coefficients)
feature_names = np.array(tfidf.get_feature_names_out())
coefs = clf.coef_[0]
top_pos = feature_names[np.argsort(coefs)[-10:]]
top_neg = feature_names[np.argsort(coefs)[:10]]


print("Top positive features:", top_pos)
print("Top negative features:", top_neg)

Top positive features: ['today' 'fun' 'loved' 'favorite' 'amazing' 'perfect' 'wonderful' 'best'
 'excellent' 'great']
Top negative features: ['worst' 'bad' 'awful' 'waste' 'boring' 'poor' 'worse' 'terrible' 'poorly'
 'dull']


In [ ]:
# inspect misclassifications
test_texts = test_df["text"].values
false_negatives = np.where((test_df["label"]==1) & (y_pred==0))[0]
false_positives = np.where((test_df["label"]==0) & (y_pred==1))[0]

print("Examples of False Negatives (true=pos, pred=neg):")
for idx in false_negatives[:3]:
    print(f"- {test_texts[idx][:200]!r}...\n")

print("Examples of False Positives (true=neg, pred=pos):")
for idx in false_positives[:3]:
    print(f"- {test_texts[idx][:200]!r}...\n")

Examples of False Negatives (true=pos, pred=neg):
- 'Its a very sensitive portrayal of life with unquenched or constrained desires. What does one do with desire in a culture and society with rigid norms? One husband finds outlet with the immigrant - sin'...

- 'This was a bold movie to hit Indian cinemas when it was released. The first movie to perhaps openly depict lesbian tendencies amongst Indian women. The leading actress of Indian cinema Shabana Azmi ad'...

- 'The theme is controversial and the depiction of the hypocritical and sexually starved india is excellent.Nothing more to this film.There is a lack of good dialogues(why was the movie in english??). Th'...

Examples of False Positives (true=neg, pred=pos):
- "First off let me say, If you haven't enjoyed a Van Damme movie since bloodsport, you probably will not like this movie. Most of these movies may not have the best plots or best actors but I enjoy thes"...

- 'Isaac Florentine has made some of the best western Martial Ar

In [ ]:
import joblib

joblib.dump(tfidf, 'tfidf_vectorizer.joblib')
joblib.dump(clf,   'tfidf_lr_model.joblib')

['tfidf_lr_model.joblib']